# 05 · Gate G2: score the test year (2019) once and freeze
Run this only after (a) notebook 04 froze the configurations and (b) the refit finished:
`python scripts/run_model.py refit --models all --h 15 60 --seeds`.
The refit already wrote the 2019 predictions (and refuses to overwrite them). This notebook only **scores** them.

In [1]:
import sys, os, json, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
from src import evaluate as ev, metrics as M, models as mo
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10, "axes.titleweight": "bold"})
def savefig(fig, name):
    fig.tight_layout(); fig.savefig(ev.FIGS / f"{name}.png", dpi=150); plt.show()
print("results folder:", ev.RESULTS, "| smoke mode:", mo.SMOKE, "| threads:", mo.N_JOBS)

results folder: C:\dev\traffic-prediction\results | smoke mode: False | threads: 11


## 0. Preconditions

In [2]:
assert (ev.RESULTS / "frozen_config.json").exists(), "freeze the configuration in notebook 04 first"
missing = [f"{n} h={h}" for n in mo.MODELS for h in ev.HORIZONS if not ev.pred_path("test", n, h).exists()]
assert not missing, f"2019 predictions missing (run the refit script): {missing}"
bb = {h: ev.best_baseline(ev.load_table(h, ["val"])) for h in ev.HORIZONS}        # chosen on VALIDATION
print("Best baseline (validation) per horizon:", bb)

AssertionError: freeze the configuration in notebook 04 first

## 1. Headline table: pooled test metrics with 95 % confidence intervals
Intervals come from resampling whole **days** (1,000 times), because errors within a day are strongly correlated.
Skill = 1 - MAE(model) / MAE(best baseline). The three baselines are scored the same way.

In [ ]:
assert (ev.RESULTS / "quality_rule.json").exists(), "run section 14 of notebook 01 first (the clean-row rule is frozen there)"
rows = []
for h in ev.HORIZONS:
    ref = ev.load_preds("test", "lgbm", h)                         # any model gives the same rows and baselines
    mk_all = np.ones(len(ref), bool)
    mk_clean = ev.clean_mask(ref)                                  # same fixed rule, applied to 2019 flow
    entries = {f"baseline: {k}": (ref[c] * ref["sc"]).to_numpy("float64") for k, c in ev.BASELINES.items()}
    for n in mo.MODELS:
        entries[mo.LABEL[n]] = ev.load_preds("test", n, h)["pred"].to_numpy("float64")
    base_all = (ref[ev.BASELINES[bb[h]]] * ref["sc"]).to_numpy("float64")
    for view, mk in (("all sensors", mk_all), ("clean rows", mk_clean)):
        day = ref["target_time"].dt.date.to_numpy()[mk]
        y = ref["y_raw"].to_numpy("float64")[mk]
        base = base_all[mk]
        for name, p in entries.items():
            p = p[mk]
            s = M.score(y, p); ci = M.day_block_bootstrap(day, y, p, base, n_boot=1000, seed=0)
            rows.append({"view": view, "rows_kept_%": mk.mean() * 100, "horizon_min": h, "model": name, "MAE": s["MAE"], "MAE_lo": ci["MAE"][0], "MAE_hi": ci["MAE"][1],
                         "RMSE": s["RMSE"], "WAPE_%": s["WAPE"] * 100, "Skill": M.skill(s["MAE"], M.mae(y, base)),
                         "Skill_lo": ci["Skill"][0], "Skill_hi": ci["Skill"][1]})
res_all = pd.DataFrame(rows)
res_all.to_csv(ev.FINAL / "headline_test_table.csv", index=False)
res = res_all[res_all["view"] == "clean rows"].copy()              # headline = clean rows; the all-sensor view is reported next to it
for h in ev.HORIZONS:
    for view in ("clean rows", "all sensors"):
        d = res_all[(res_all.horizon_min == h) & (res_all.view == view)]
        print(f"\n=== 2019, horizon {h} min, {view} ({d['rows_kept_%'].iloc[0]:.1f}% of rows; skill vs {bb[h]}) ===")
        display(d.drop(columns=["horizon_min", "view", "rows_kept_%"]).sort_values("MAE").round(3).reset_index(drop=True))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, h in zip(axes, ev.HORIZONS):
    d = res[res.horizon_min == h].sort_values("MAE", ascending=False)
    col = ["#999999" if m.startswith("baseline") else "#0072B2" for m in d["model"]]
    ax.barh(d["model"], d["MAE"], xerr=[d["MAE"] - d["MAE_lo"], d["MAE_hi"] - d["MAE"]], color=col, capsize=2)
    ax.set_xlabel("2019 MAE (vehicles per 15 min), 95% CI"); ax.set_title(f"{h}-minute horizon")
savefig(fig, "test_headline")

## 2. Warm vs cold sensors (do models generalise to sensors never seen in training?)

In [ ]:
rows = []
for h in ev.HORIZONS:
    for n in mo.MODELS:
        d = ev.load_preds("test", n, h)
        for role, g in d.groupby("role"):
            rows.append({"horizon_min": h, "model": mo.LABEL[n], "role": role, "MAE": M.mae(g["y_raw"], g["pred"]), "WAPE_%": M.wape(g["y_raw"], g["pred"]) * 100})
wc = pd.DataFrame(rows)
wc.to_csv(ev.FINAL / "test_warm_vs_cold.csv", index=False)
piv = wc.pivot_table(index=["horizon_min", "model"], columns="role", values="WAPE_%")
piv["cold_minus_warm_pts"] = piv["cold"] - piv["warm"]
display(piv.round(2))

## 3. By quarter of 2019 and by corridor (best 3 models at each horizon)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 7))
for r, h in enumerate(ev.HORIZONS):
    top3 = res[(res.horizon_min == h) & (~res.model.str.startswith("baseline"))].nsmallest(3, "MAE")["model"].tolist()
    inv = {v: k for k, v in mo.LABEL.items()}
    for n in top3:
        d = ev.load_preds("test", inv[n], h)
        q = d.groupby(d["target_time"].dt.quarter).apply(lambda x: M.wape(x["y_raw"], x["pred"]) * 100)
        c = d.groupby("corridor").apply(lambda x: M.wape(x["y_raw"], x["pred"]) * 100)
        axes[r, 0].plot(q.index, q.values, "o-", label=n); axes[r, 1].plot(range(len(c)), c.values, "o-", label=n)
    axes[r, 0].set_xticks([1, 2, 3, 4]); axes[r, 0].set_xlabel("quarter of 2019"); axes[r, 0].set_ylabel("WAPE (%)"); axes[r, 0].set_title(f"{h} min, by quarter"); axes[r, 0].legend()
    axes[r, 1].set_xticks(range(len(c))); axes[r, 1].set_xticklabels(c.index, fontsize=8); axes[r, 1].set_title(f"{h} min, by corridor")
savefig(fig, "test_quarter_corridor")

## 4. Per-sensor spread (is the average hiding bad sensors?)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, h in zip(axes, ev.HORIZONS):
    order = res[(res.horizon_min == h) & (~res.model.str.startswith("baseline"))].sort_values("MAE")["model"].tolist()
    inv = {v: k for k, v in mo.LABEL.items()}
    data = []
    for n in order:
        d = ev.load_preds("test", inv[n], h)
        data.append(d.groupby("ID2").apply(lambda x: M.wape(x["y_raw"], x["pred"]) * 100).to_numpy())
    ax.boxplot(data, showfliers=False); ax.set_xticks(range(1, len(order) + 1)); ax.set_xticklabels(order); ax.tick_params(axis="x", rotation=45)
    ax.set_ylabel("per-sensor WAPE (%)"); ax.set_title(f"{h}-minute horizon")
savefig(fig, "test_per_sensor")

## 5. LightGBM seed stability and compute cost

In [ ]:
rows = []
for h in ev.HORIZONS:
    maes = []
    for tag in ["", "_seed43", "_seed44"]:
        if ev.pred_path("test", "lgbm", h, tag).exists():
            d = ev.load_preds("test", "lgbm", h, tag); maes.append(M.mae(d["y_raw"], d["pred"]))
    rows.append({"horizon_min": h, "seeds": len(maes), "MAE_mean": np.mean(maes), "MAE_sd": np.std(maes, ddof=1) if len(maes) > 1 else np.nan})
display(pd.DataFrame(rows).round(4))
cost = pd.DataFrame([json.loads(f.read_text()) for f in sorted(ev.FINAL.glob("cost_*_h*.json")) if "seed" not in f.name])
cost["model"] = cost["model"].map(mo.LABEL)
cost = cost[["model", "h", "fit_rows", "n_estimators", "fit_s", "predict_s_per_100k", "model_MB"]].sort_values(["h", "fit_s"])
cost.to_csv(ev.FINAL / "cost_table.csv", index=False)
display(cost.round(2))

## 6. Freeze (Gate G2)
Stores a fingerprint of every 2019 prediction file and the headline table, so you can prove later that nothing
changed after the test year was scored.

In [ ]:
import hashlib, datetime
def sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:16]
frz = {"frozen_at": datetime.datetime.now().isoformat(timespec="seconds"),
       "best_baseline_by_validation": {str(h): b for h, b in bb.items()},
       "test_prediction_files": {f.name: sha(f) for f in sorted(ev.PREDS.glob("test_*.parquet"))},
       "frozen_config": sha(ev.RESULTS / "frozen_config.json"), "headline_table": sha(ev.FINAL / "headline_test_table.csv")}
(ev.RESULTS / "FROZEN.json").write_text(json.dumps(frz, indent=2))
print("GATE G2: results frozen at", frz["frozen_at"], "-", len(frz["test_prediction_files"]), "prediction files fingerprinted")